In [1]:
%run _BaseInfo.ipynb
import requests
import pandas as pd
import numpy as np
import os
import json
import time
from datetime import datetime, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

os.makedirs('./Result/fund', exist_ok=True)

# ── session ───────────────────────────────────────────
_session = requests.Session()
_retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET']
)
_session.mount('https://', HTTPAdapter(max_retries=_retry))
_session.mount('http://',  HTTPAdapter(max_retries=_retry))
_headers = {'User-Agent': 'Mozilla/5.0'}

N_DAYS = 5  # 預設分析天數

[上市] 線上抓取成功，共 1090 筆
[上櫃] 線上抓取成功，共 891 筆
[output] 儲存成功，共 1981 筆
df_info 共 1979 筆，上市：1089，上櫃：890


## 日期 / 快取工具

In [2]:
def _to_roc(date_str):
    """YYYYMMDD → 民國YYY/MM/DD（TPEX 日期格式）"""
    dt = datetime.strptime(date_str, '%Y%m%d')
    return f'{dt.year - 1911}/{dt.month:02d}/{dt.day:02d}'


def _recent_weekdays(n=60):
    """回傳最近 n 個工作日（含今天），由新到舊"""
    days, d = [], datetime.today()
    while len(days) < n:
        if d.weekday() < 5:
            days.append(d.strftime('%Y%m%d'))
        d -= timedelta(days=1)
    return days


def _cache_path(date_str):
    return f'./Result/fund/{date_str}.json'

## 經取函式

In [3]:
def _fetch_twse(date_str):
    """
    上市三大法人買賣超（T86）
    回傳 list[dict]，無資料時回傳 None
    """
    url = (
        f'https://www.twse.com.tw/rwd/zh/fund/T86'
        f'?date={date_str}&selectType=ALL&response=json'
    )
    try:
        r = _session.get(url, headers=_headers, timeout=15)
        data = r.json()
        if data.get('stat') != 'OK' or not data.get('data'):
            return None
        fields = data['fields']
        return [dict(zip(fields, row)) for row in data['data']]
    except Exception as e:
        print(f'  [TWSE] {date_str} 失敗: {e}')
        return None


def _fetch_tpex(date_str):
    """
    上櫃三大法人買賣超
    回傳 list[dict]，無資料時回傳 None
    """
    url = (
        f'https://www.tpex.org.tw/web/stock/3insti/daily_trade/3itrade_hedge_result.php'
        f'?l=zh-tw&o=json&d={_to_roc(date_str)}&se=EW&t=D'
    )
    tpex_headers = {**_headers, 'Referer': 'https://www.tpex.org.tw/'}
    try:
        r = _session.get(url, headers=tpex_headers, timeout=15)
        r.encoding = 'utf-8'
        data = r.json()

        # --- 嘗試各種可能的 response 格式 ---

        # 格式 A：新版 tables 結構
        #   { "tables": [{ "fields": [...], "data": [[...], ...] }] }
        if 'tables' in data:
            tables = data['tables']
            if tables and isinstance(tables, list):
                tbl = tables[0]
                fields = tbl.get('fields') or tbl.get('header') or []
                rows_raw = tbl.get('data') or tbl.get('aaData') or []
                if rows_raw and fields:
                    n = len(fields)
                    rows = []
                    for row in rows_raw:
                        if not isinstance(row, list):
                            continue
                        d = dict(zip(fields, row))
                        # 新版 24 欄格式欄位名稱含括號（如「外資及陸資(不含外資自營商)買賣超股數」），
                        # 導致 _load_day 以舊欄位名稱 .get() 取不到值。
                        # 用 setdefault 補齊 _load_day 預期的欄位名稱（key 已存在則不覆蓋）。
                        if n >= 24 and len(row) >= 24:
                            d.setdefault('外資及陸資買賣超股數',  row[4])   # 不含外資自營商
                            d.setdefault('外資自營商買賣超股數',  row[7])
                            d.setdefault('投信買進股數',          row[11])
                            d.setdefault('投信賣出股數',          row[12])
                            d.setdefault('投信買賣超股數',        row[13])
                            d.setdefault('自營商買賣超股數',      row[22])  # 自行+避險合計
                            d.setdefault('三大法人買賣超股數',    row[23])
                        rows.append(d)
                    return rows or None
                elif rows_raw and isinstance(rows_raw[0], dict):
                    return rows_raw
            print(f'  [TPEX] {date_str} tables 結構無法解析: {str(tables)[:200]}')
            return None

        # 格式 B：舊版 aaData 結構
        aa = data.get('aaData') or data.get('data') or []
        if aa:
            cols = [
                '代號', '名稱',
                '外資及陸資買進股數', '外資及陸資賣出股數', '外資及陸資買賣超股數',
                '外資自營商買進股數', '外資自營商賣出股數', '外資自營商買賣超股數',
                '投信買進股數', '投信賣出股數', '投信買賣超股數',
                '自營商買賣超股數',
                '自營商買進股數(自行買賣)', '自營商賣出股數(自行買賣)', '自營商買賣超股數(自行買賣)',
                '自營商買進股數(避險)',   '自營商賣出股數(避險)',   '自營商買賣超股數(避險)',
                '三大法人買賣超股數'
            ]
            rows = []
            for row in aa:
                if isinstance(row, dict):
                    rows.append(row)
                elif isinstance(row, list):
                    n = min(len(row), len(cols))
                    rows.append(dict(zip(cols[:n], row[:n])))
            return rows or None

        print(f'  [TPEX] {date_str} 無資料，keys: {list(data.keys())}')
        return None
    except Exception as e:
        print(f'  [TPEX] {date_str} 失敗: {e}')
        return None


In [4]:
# ── TPEX 結構 debug：確認所有欄位名稱 ──
import requests as _req
_roc = _to_roc('20260623')
_url = (
    f'https://www.tpex.org.tw/web/stock/3insti/daily_trade/3itrade_hedge_result.php'
    f'?l=zh-tw&o=json&d={_roc}&se=EW&t=D'
)
_r = _req.get(_url, headers={**_headers, 'Referer': 'https://www.tpex.org.tw/'}, timeout=15)
_data = _r.json()
_tbl = _data['tables'][0]
_fields = _tbl['fields']
_row0   = _tbl['data'][0]
print(f'共 {len(_fields)} 個欄位（columnNum={_data["columnNum"]}）\n')
for i, (f, v) in enumerate(zip(_fields, _row0)):
    print(f'  [{i:02d}] {f!r:45s} = {v!r}')


共 24 個欄位（columnNum=25）

  [00] '代號'                                          = '006201'
  [01] '名稱'                                          = '元大富櫃50'
  [02] '買進股數'                                        = '46,030'
  [03] '賣出股數'                                        = '2,000'
  [04] '買賣超股數'                                       = '44,030'
  [05] '買進股數'                                        = '0'
  [06] '賣出股數'                                        = '0'
  [07] '買賣超股數'                                       = '0'
  [08] '買進股數'                                        = '46,030'
  [09] '賣出股數'                                        = '2,000'
  [10] '買賣超股數'                                       = '44,030'
  [11] '買進股數'                                        = '0'
  [12] '賣出股數'                                        = '55,000'
  [13] '買賣超股數'                                       = '-55,000'
  [14] '買進股數'                                        = '0'
  [15] '賣出股數'                             

## 主爬蟲：crawl_fund_n_days(n)
- 已存快取 → 直接略過（不重爬）
- 無資料（假日 / 未開盤）→ 自動跳過，不計入 n
- 每日資料存於 `Result/fund/YYYYMMDD.json`

In [5]:
def crawl_fund_n_days(n=N_DAYS):
    """
    爬取最近 n 個交易日的三大法人資料。
    已爬過的日期直接讀快取，不重新下載。
    回傳：已收集的日期列表（由新到舊）
    """
    today      = datetime.today().strftime('%Y%m%d')
    candidates = _recent_weekdays(n * 4)
    collected  = []

    for date_str in candidates:
        if len(collected) >= n:
            break
        if date_str > today:
            continue

        cache = _cache_path(date_str)

        if os.path.exists(cache):
            print(f'[快取] {date_str}')
            collected.append(date_str)
            continue

        print(f'[爬取] {date_str}...', end=' ')
        twse = _fetch_twse(date_str)
        time.sleep(0.5)
        tpex = _fetch_tpex(date_str)
        time.sleep(0.5)

        if twse is None and tpex is None:
            print('無資料（假日或未開盤），略過')
            continue

        payload = {
            'date': date_str,
            'twse': twse or [],
            'tpex': tpex or []
        }
        with open(cache, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False)

        print(f'TWSE {len(twse or []):,} 筆 / TPEX {len(tpex or []):,} 筆 → 已存檔')
        collected.append(date_str)

    if collected:
        print(f'\n✅ 共收集 {len(collected)} 個交易日：{collected[-1]} ~ {collected[0]}')
    else:
        print('⚠️  未收集到任何資料')
    return collected

## 讀取 / 正規化

In [6]:
_NUM_COLS = [
    'foreign_net', 'foreign_dealer_net',
    'trust_buy', 'trust_sell', 'trust_net',
    'dealer_net', 'total_net'
]


def _load_day(date_str):
    """讀取單日快取，回傳正規化後的 DataFrame"""
    cache = _cache_path(date_str)
    if not os.path.exists(cache):
        return pd.DataFrame()

    with open(cache, 'r', encoding='utf-8') as f:
        data = json.load(f)

    rows = []

    for d in data.get('twse', []):
        rows.append({
            'date':               date_str,
            'code':               d.get('證券代號', '').strip(),
            'name':               d.get('證券名稱', '').strip(),
            'market':             '上市',
            'foreign_net':        d.get('外陸資買賣超股數(不含外資自營商)', '0'),
            'foreign_dealer_net': d.get('外資自營商買賣超股數', '0'),
            'trust_buy':          d.get('投信買進股數', '0'),
            'trust_sell':         d.get('投信賣出股數', '0'),
            'trust_net':          d.get('投信買賣超股數', '0'),
            'dealer_net':         d.get('自營商買賣超股數', '0'),
            'total_net':          d.get('三大法人買賣超股數', '0'),
        })

    for d in data.get('tpex', []):
        rows.append({
            'date':               date_str,
            'code':               d.get('代號', '').strip(),
            'name':               d.get('名稱', '').strip(),
            'market':             '上櫃',
            'foreign_net':        d.get('外資及陸資買賣超股數', '0'),
            'foreign_dealer_net': d.get('外資自營商買賣超股數', '0'),
            'trust_buy':          d.get('投信買進股數', '0'),
            'trust_sell':         d.get('投信賣出股數', '0'),
            'trust_net':          d.get('投信買賣超股數', '0'),
            'dealer_net':         d.get('自營商買賣超股數', '0'),
            'total_net':          d.get('三大法人買賣超股數', '0'),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    for col in _NUM_COLS:
        df[col] = (
            pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False).str.strip(),
                errors='coerce'
            ).fillna(0).astype(int)
        )

    df = df[df['code'].str.match(r'^\d{4,6}$', na=False)].copy()
    return df

## 分析：analyze_fund_n_days(n)
讀取最近 n 個交易日快取，輸出：
1. **df_volume** — 各股外資 / 投信累計買賣超量
2. **df_streak** — 各股外資 / 投信最近連續買超或賣超天數
   - 正數 = 連續買超 N 天；負數 = 連續賣超 N 天

In [7]:
def _calc_streak(series):
    """
    計算序列末端連續同向天數。
    NaN 視為 0（該日無交易資料）。
    正數 = 連買超 N 天；負數 = 連賣超 N 天；0 = 當日持平。
    """
    vals = series.fillna(0).values
    if len(vals) == 0:
        return 0
    last_sign = int(np.sign(vals[-1]))
    if last_sign == 0:
        return 0
    streak = 0
    for v in reversed(vals):
        if int(np.sign(v)) == last_sign:
            streak += 1
        else:
            break
    return streak * last_sign


def analyze_fund_n_days(n=N_DAYS):
    """
    讀取最近 n 個交易日的快取，回傳三個 DataFrame：
      df_volume  — 累計買賣量彙總
      df_streak  — 連續買賣超天數（外資 / 投信）
      df_result  — 兩者合併後的完整結果
    """
    dates = []
    for d in _recent_weekdays(n * 4):
        if os.path.exists(_cache_path(d)):
            dates.append(d)
        if len(dates) >= n:
            break

    if not dates:
        print('⚠️  無快取資料，請先執行 crawl_fund_n_days()')
        return None, None, None

    dates = sorted(dates)
    print(f'分析期間：{dates[0]} ~ {dates[-1]}（共 {len(dates)} 個交易日）')

    all_df = pd.concat([_load_day(d) for d in dates], ignore_index=True)

    # 1. 累計買賣量
    df_volume = (
        all_df
        .groupby(['code', 'name', 'market'], sort=False)
        .agg(
            **{'外資_累計買賣超':  ('foreign_net',  'sum')},
            **{'投信_累計買進':    ('trust_buy',    'sum')},
            **{'投信_累計賣出':    ('trust_sell',   'sum')},
            **{'投信_累計買賣超':  ('trust_net',    'sum')},
            **{'自營商_累計買賣超': ('dealer_net',   'sum')},
            **{'三大法人_累計買賣超': ('total_net',  'sum')},
            **{'交易日數':         ('date',         'count')},
        )
        .reset_index()
        .sort_values('三大法人_累計買賣超', ascending=False)
        .reset_index(drop=True)
    )

    # 2. 連續天數
    key_cols = ['code', 'name', 'market']

    pivot_f = (
        all_df
        .pivot_table(index=key_cols, columns='date',
                     values='foreign_net', aggfunc='sum')
        .sort_index(axis=1)
    )
    pivot_t = (
        all_df
        .pivot_table(index=key_cols, columns='date',
                     values='trust_net', aggfunc='sum')
        .sort_index(axis=1)
    )

    streak_f = pivot_f.apply(_calc_streak, axis=1).rename('外資連續天數')
    streak_t = pivot_t.apply(_calc_streak, axis=1).rename('投信連續天數')

    df_streak = (
        pd.concat([streak_f, streak_t], axis=1)
        .reset_index()
        .sort_values('外資連續天數', ascending=False)
        .reset_index(drop=True)
    )

    # 3. 合併結果
    df_result = (
        df_volume
        .merge(df_streak[key_cols + ['外資連續天數', '投信連續天數']],
               on=key_cols, how='left')
    )

    return df_volume, df_streak, df_result


---
## 執行：爬取最近 N_DAYS 個交易日

In [8]:
collected = crawl_fund_n_days(n=N_DAYS)

[爬取] 20260624... TWSE 13,801 筆 / TPEX 915 筆 → 已存檔
[快取] 20260623
[快取] 20260622
[爬取] 20260619...   [TPEX] 20260619 tables 結構無法解析: [{'title': '三大法人買賣明細資訊', 'date': '115/06/19', 'fields': ['代號', '名稱', '買進股數', '賣出股數', '買賣超股數', '買進股數', '賣出股數', '買賣超股數', '買進股數', '賣出股數', '買賣超股數', '買進股數', '賣出股數', '買賣超股數', '買進股數', '賣出股數', '買賣超股數', '買進股數'
無資料（假日或未開盤），略過
[快取] 20260618
[快取] 20260617

✅ 共收集 5 個交易日：20260617 ~ 20260624


## 執行：分析結果

In [9]:
df_volume, df_streak, df_result = analyze_fund_n_days(n=N_DAYS)
df_result

分析期間：20260617 ~ 20260624（共 5 個交易日）


,code,name,market,外資_累計買賣超,投信_累計買進,投信_累計賣出,投信_累計買賣超,自營商_累計買賣超,三大法人_累計買賣超,交易日數,外資連續天數,投信連續天數
0,00919,群益台灣精選高息,上市,153861181,0,0,0,52197125,206058306,5,-1,0
1,2610,華航,上市,131622297,1195000,433080,761920,15809724,148193941,5,2,5
2,2618,長榮航,上市,141662389,2393684,21211934,-18818250,4740901,127585040,5,2,1
3,2890,永豐金,上市,-9065417,109472259,167879,109304380,-380612,99858351,5,-3,5
4,6770,力積電,上市,79622499,3849000,2000,3847000,8671663,92141162,5,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...
21171,6116,彩晶,上市,-101992270,2362000,0,2362000,-3758232,-103388502,5,1,0
21172,3481,群創,上市,-206764877,41077019,473666,40603353,13124881,-153036643,5,-5,4
21173,0050,元大台灣50,上市,-135504058,12497000,0,12497000,-61710991,-184718049,5,-2,5
21174,009821,野村稀土關鍵資源,上市,-4311720,0,0,0,-266480829,-270792549,5,-1,0


## 快速篩選範例

In [10]:
# 外資連買 >= 3 天 且 投信也連買
df_both_buy = df_result[
    (df_result['外資連續天數'] >= 3) &
    (df_result['投信連續天數'] > 0)
].copy()
print(f'外資連買≥3天 且 投信連買：{len(df_both_buy)} 檔')
df_both_buy

外資連買≥3天 且 投信連買：10 檔


,code,name,market,外資_累計買賣超,投信_累計買進,投信_累計賣出,投信_累計買賣超,自營商_累計買賣超,三大法人_累計買賣超,交易日數,外資連續天數,投信連續天數
60,8054,安國,上櫃,9177887,20000,0,20000,477389,9675276,5,5,1
66,2363,矽統,上市,6925530,70000,0,70000,874886,7870416,5,4,2
169,4722,國精化,上市,687596,2408000,184000,2224000,53137,2964733,5,3,1
242,1560,中砂,上市,1870899,157000,10000,147000,10204,2028103,5,5,4
263,6831,邁科,上市,1585664,326000,0,326000,4630,1916294,5,5,3
797,5236,凌陽創新,上櫃,690871,5000,0,5000,7800,703671,5,5,1
1027,6526,達發,上市,395249,141000,5000,136000,10549,541798,5,5,5
1541,5269,祥碩,上市,281577,30056,7027,23029,37380,341986,5,5,3
19674,4991,環宇-KY,上櫃,448620,105000,1258720,-1153720,170661,-534439,5,4,1
19947,4966,譜瑞-KY,上櫃,302426,27000,1007450,-980450,20474,-657550,5,3,1


In [11]:
# 三大法人累計買超 Top 20
df_result.head(20)

,code,name,market,外資_累計買賣超,投信_累計買進,投信_累計賣出,投信_累計買賣超,自營商_累計買賣超,三大法人_累計買賣超,交易日數,外資連續天數,投信連續天數
0,00919,群益台灣精選高息,上市,153861181,0,0,0,52197125,206058306,5,-1,0
1,2610,華航,上市,131622297,1195000,433080,761920,15809724,148193941,5,2,5
2,2618,長榮航,上市,141662389,2393684,21211934,-18818250,4740901,127585040,5,2,1
3,2890,永豐金,上市,-9065417,109472259,167879,109304380,-380612,99858351,5,-3,5
4,6770,力積電,上市,79622499,3849000,2000,3847000,8671663,92141162,5,1,2
5,2002,中鋼,上市,365780565,3322035,289748592,-286426557,1755963,81109971,5,1,0
6,2409,友達,上市,61533819,167234,244710,-77476,17057807,78514150,5,-2,2
7,1301,台塑,上市,179834523,1215014,108680854,-107465840,1661996,74030679,5,1,0
8,2449,京元電子,上市,71814572,3236792,16582082,-13345290,1695979,60165261,5,4,-5
9,2892,第一金,上市,-30631092,90363316,438125,89925191,785278,60079377,5,-4,5


In [12]:
# 匯出結果到 Result/fund/
from datetime import datetime as _dt
_today = _dt.today().strftime('%Y%m%d')

df_result.to_excel(f'Result/fund/fund_result_{_today}.xlsx', index=False)
print(f'已輸出 Result/fund/fund_result_{_today}.xlsx')

已輸出 Result/fund/fund_result_20260624.xlsx
